# Session 12: Semantic Segmentation

**CVI4IC - Summer Semester 2026**

Pixel-level classification using deep learning models.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torchvision import transforms
from torchvision.models.segmentation import deeplabv3_resnet50
import cv2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Dice Loss

In [ ]:
def dice_loss(pred, target, smooth=1e-6):
    """Compute Dice Loss for binary segmentation."""
    pred_flat = pred.reshape(-1)
    target_flat = target.reshape(-1)
    
    intersection = (pred_flat * target_flat).sum()
    dice = (2.0 * intersection + smooth) / (pred_flat.sum() + target_flat.sum() + smooth)
    
    return 1.0 - dice

# Example
pred = np.zeros((100, 100), dtype=np.float32)
pred[20:80, 20:80] = 1.0

target = np.zeros((100, 100), dtype=np.float32)
target[30:90, 30:90] = 1.0

loss = dice_loss(pred, target)
print(f"Dice Loss: {loss:.4f}")
print(f"Dice Score: {1.0 - loss:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(pred, cmap="gray")
axes[0].set_title("Prediction")
axes[1].imshow(target, cmap="gray")
axes[1].set_title("Ground Truth")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 2. Simple U-Net

In [ ]:
class MiniUNet(nn.Module):
    """A minimal U-Net for demonstration."""
    def __init__(self, in_channels=1, num_classes=2):
        super().__init__()
        # Encoder
        self.enc1 = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU()
        )
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU()
        )
        self.pool2 = nn.MaxPool2d(2)
        
        # Bottleneck
        self.bottleneck = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.ReLU()
        )
        
        # Decoder
        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = nn.Sequential(
            nn.Conv2d(128, 64, 3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU()
        )
        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec1 = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU()
        )
        self.final = nn.Conv2d(32, num_classes, 1)
    
    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        
        # Bottleneck
        b = self.bottleneck(self.pool2(e2))
        
        # Decoder with skip connections
        d2 = self.dec2(torch.cat([self.up2(b), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        
        return self.final(d1)

model = MiniUNet(in_channels=1, num_classes=2).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Mini U-Net parameters: {total_params:,}")

# Test forward pass
x = torch.randn(1, 1, 64, 64).to(device)
out = model(x)
print(f"Input shape: {x.shape} -> Output shape: {out.shape}")

## 3. Pretrained DeepLabV3

In [ ]:
# Load pretrained DeepLabV3
deeplab = deeplabv3_resnet50(weights="DEFAULT")
deeplab.eval()
deeplab.to(device)

# Create a dummy color image for testing
test_img = np.random.randint(0, 255, (480, 640, 3), dtype=np.uint8)

# Preprocess
preprocess = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

input_tensor = preprocess(test_img).unsqueeze(0).to(device)

with torch.no_grad():
    output = deeplab(input_tensor)["out"][0]

predictions = output.argmax(0).cpu().numpy()
print(f"Output classes: {np.unique(predictions)}")
print(f"Prediction map shape: {predictions.shape}")

plt.figure(figsize=(8, 6))
plt.imshow(predictions, cmap="tab20")
plt.title("DeepLabV3 Segmentation")
plt.colorbar()
plt.show()

## 4. Evaluation Metrics

In [ ]:
def pixel_accuracy(pred, target):
    """Compute pixel-level accuracy."""
    return (pred == target).sum() / target.size

def mean_iou(pred, target, num_classes):
    """Compute mean IoU across classes."""
    ious = []
    for cls in range(num_classes):
        pred_mask = (pred == cls)
        target_mask = (target == cls)
        intersection = (pred_mask & target_mask).sum()
        union = (pred_mask | target_mask).sum()
        if union > 0:
            ious.append(intersection / union)
    return np.mean(ious) if ious else 0.0

# Example
pred_map = np.random.randint(0, 3, (100, 100))
gt_map = np.random.randint(0, 3, (100, 100))

print(f"Pixel Accuracy: {pixel_accuracy(pred_map, gt_map):.4f}")
print(f"Mean IoU: {mean_iou(pred_map, gt_map, 3):.4f}")

## Exercises

1. Train the Mini U-Net on synthetic binary segmentation data (circles/squares)
2. Run DeepLabV3 on real images and visualize the segmentation masks
3. Implement and compare Cross-Entropy Loss vs. Dice Loss for segmentation training

In [ ]:
# Your code here
